# --- Model Train Orchestrator ---

In [1]:
# ===================
# Setup
# ===================
import os
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import torch

PROJECT_ROOT = Path(os.getcwd()).parent

# Connect to custom-defined modules
sys.path.append(str(PROJECT_ROOT))

# Auto-reload scripts if any changes
%reload_ext autoreload
%autoreload 2

In [2]:
# ========================
# Custom Modules
# ========================
from src.data.dataloader import create_multitask_dataloader

from src.models.config import MultiTaskModelConfig
from src.models.network import MultiTaskYOLO
from src.models.loss import MultiTaskUncertaintyLoss

from src.training.trainer import MultiTaskTrainer

In [3]:
# =================================
# Run DataLoader Verification Cell
# =================================

# Configure paths to the split dataset directories
data_config = {
    "turnaround": {
        "images": str(PROJECT_ROOT / "data/processed/train/turnaround/images"),
        "labels": str(PROJECT_ROOT / "data/processed/train/turnaround/labels")
    },
    "ppe": {
        "images": str(PROJECT_ROOT / "data/processed/train/ppe/images"),
        "labels": str(PROJECT_ROOT / "data/processed/train/ppe/labels")
    },
    "fod": {
        "images": str(PROJECT_ROOT / "data/processed/train/fod/images"),
        "labels": str(PROJECT_ROOT / "data/processed/train/fod/labels")
    }
}

train_loader = create_multitask_dataloader(
    data_dirs=data_config,
    batch_size=8,
    shuffle=True,
    num_workers=2
)

# Fetch one batch to verify
for images, task_targets, task_ids in train_loader:
    print("✅ DataLoader Batch Ready!")
    print(f"• Images Tensor: {images.shape}")
    print(f"• Task IDs: {task_ids.tolist()}")
    break

📦 Loaded [TURNAROUND] dataset: 6813 samples
📦 Loaded [PPE] dataset: 5641 samples
📦 Loaded [FOD] dataset: 23655 samples
✅ Combined Multi-Task Dataset total size: 36109 samples

✅ DataLoader Batch Ready!
• Images Tensor: torch.Size([8, 3, 640, 640])
• Task IDs: [2, 1, 2, 0, 2, 2, 0, 2]


In [ ]:
# ==========================
# Run a 1-Epoch Dry Run
# ==========================

# 1. Config & Model
config = MultiTaskModelConfig()
model = MultiTaskYOLO(config)
loss_fn = MultiTaskUncertaintyLoss(model, config)

# 2. Optimizer & Scheduler
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(loss_fn.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

# 3. Trainer Instance
trainer = MultiTaskTrainer(
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=train_loader, # Pass train_loader temporarily for sanity check
    optimizer=optimizer,
    config=config,
    use_wandb=False,
    exp_name="dry_run_test"
)

# 4. Launch 1 Dry-Run Epoch
trainer.fit(num_epochs=1)

🚀 Starting Multi-Task Model Training on device [CPU] for 1 Epcohs...



Epoch 1 [Train]:   0%|          | 0/4514 [00:00<?, ?it/s]